# 06 — Gates as Unitary Evolution

Meet the operations that *move* a qubit — and discover why preserving the inner product forces every one of them to be unitary.

**Prerequisites:** `01/05_inner_product_and_dirac`, `01/03_the_bloch_sphere`.

**What you'll be able to do**
- Describe a quantum gate as a matrix $U$ acting on a state, $|\psi'\rangle = U|\psi\rangle$.
- Explain why a gate must be **unitary** ($U^\dagger U = I$) — and check it with `is_unitary`.
- See that a single-qubit gate is a rotation of the Bloch sphere, and that gates are reversible.

So far we have *described* states and *measured* them. To compute, we need to **change** them — to push $|0\rangle$ into a superposition, to rotate one state into another. Every such operation is a quantum gate, and today you learn the one rule they all obey, straight from the inner product you built in `01/05`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, bloch_vector
from qot_course.quantum.gates import apply_gate, is_unitary

np.random.seed(42)
viz.use_course_style()

## 1. A gate is a matrix acting on a state

A single-qubit gate is a $2\times2$ matrix $U$. Applying it means matrix-vector multiplication:

$$ |\psi'\rangle = U\,|\psi\rangle. $$

Here is a concrete one — a rotation that tips $|0\rangle$ partway toward $|1\rangle$. (We do not name it yet; the famous named gates arrive in `01/07`-`01/09`. For now it is a stand-in for "some operation".)

In [ ]:
theta = 2 * np.pi / 3
U = np.array([[np.cos(theta / 2), -np.sin(theta / 2)],
              [np.sin(theta / 2),  np.cos(theta / 2)]], dtype=complex)

out = apply_gate(U, KET_0)
print("U =\n", np.round(U, 3))
print("input  |0>     =", np.round(KET_0, 3))
print("output U|0>    =", np.round(out, 3))
print("output norm    =", round(float(np.linalg.norm(out)), 6), " (still 1)")

**Read the output.** The matrix $U$ turned the sharp state $|0\rangle$ into a superposition — and the output still has norm $1$. That last fact is not an accident. A state must stay normalised after *any* gate, or the Born-rule probabilities would no longer sum to $1$. That single requirement pins down what matrices are allowed.

## 2. Why gates are unitary

A gate must preserve the inner product for *every* pair of states (so all overlaps, and hence all probabilities, survive). Writing that requirement out:

$$ \langle U\varphi | U\psi\rangle = \langle\varphi|\,U^\dagger U\,|\psi\rangle = \langle\varphi|\psi\rangle \quad\text{for all } \varphi,\psi \;\Longleftrightarrow\; U^\dagger U = I. $$

A matrix with $U^\dagger U = I$ is called **unitary**. So "valid gate" and "unitary matrix" are the same thing. Let's check our $U$, and then watch a *non*-unitary matrix break the rule.

In [ ]:
print("U is unitary?      ", is_unitary(U))
print("U^dagger U =\n", np.round(U.conj().T @ U, 6))

# A non-unitary matrix: it does NOT preserve the norm, so it is not a valid gate.
M = np.array([[1.0, 1.0], [0.0, 1.0]], dtype=complex)
bad = apply_gate(M, KET_0 / np.sqrt(1))   # apply to a normalised state
print("\nM is unitary?      ", is_unitary(M))
some_state = np.array([1, 1], dtype=complex) / np.sqrt(2)
print("norm before M       :", round(float(np.linalg.norm(some_state)), 3))
print("norm after  M       :", round(float(np.linalg.norm(apply_gate(M, some_state))), 3), " (!= 1: not a valid gate)")

**Read the output.** For $U$, the product $U^\dagger U$ is the identity — it is unitary, a legitimate gate. The shear matrix $M$ is not: it stretches a normalised state to a norm above $1$, which would make probabilities exceed $1$. Nature forbids it as an evolution. Unitarity is the gatekeeper.

## 3. Unitary means reversible

Because $U^\dagger U = I$, the inverse of a gate is its conjugate transpose: $U^{-1} = U^\dagger$. Every gate can be *undone* by another gate — quantum evolution is reversible. (Measurement is the opposite: it discards information and cannot be reversed. We will feel that contrast in `01/10`.)

In [ ]:
recovered = apply_gate(U.conj().T, out)     # apply U-dagger to undo U
print("U applied then U-dagger:", np.round(recovered, 6))
print("back to |0>?           ", np.allclose(recovered, KET_0))

**Read the output.** Applying $U^\dagger$ after $U$ returns the state exactly to $|0\rangle$. No information was lost on the way out and back — the hallmark of unitary evolution.

## 4. A gate is a rotation of the Bloch sphere

On the Bloch sphere from `01/03`, a single-qubit gate moves a point to another point *on the same sphere* — it is a rotation. Our $U$ took the north pole $|0\rangle$ and swung it down to the Bloch latitude $\theta$. Let's see where it landed.

In [ ]:
r_in, r_out = bloch_vector(KET_0), bloch_vector(out)
print("Bloch vector of |0>   :", np.round(r_in, 3), " (north pole)")
print("Bloch vector of U|0>  :", np.round(r_out, 3))
fig = viz.plot_bloch(out, title=r"$U|0\rangle$: the gate rotated the state on the sphere")
plt.show()

**Read the figure.** The output sits on the Bloch sphere at a new latitude — the gate *rotated* $|0\rangle$ rather than shrinking or stretching it (which is exactly what staying on the sphere means, and what unitarity guarantees). Every single-qubit gate you meet from here on is one of these rotations; the named gates of the next notebooks are the most useful rotation angles and axes.

## Your turn

1. **Your own gate.** Build $U_\varphi = \mathrm{diag}(1,\,e^{i\varphi})$ for $\varphi = \pi/2$. Confirm it is unitary, apply it to $\tfrac{1}{\sqrt2}(|0\rangle+|1\rangle)$, and describe what it does on the Bloch sphere (hint: it spins the longitude).
2. **Measurement is not a gate.** Argue why projecting onto $|0\rangle$ (forcing the outcome $0$) cannot be written as a unitary. Which property of section 3 does it violate?
3. **Composition.** If $U$ and $V$ are both unitary, show $VU$ is unitary too. (Applying gates in sequence is multiplying their matrices — and the product stays a valid gate.)

## What you built

- You applied a gate as a matrix, $|\psi'\rangle = U|\psi\rangle$, and saw it keep the state normalised.
- You derived, from preserving the inner product, that every gate is **unitary** ($U^\dagger U = I$) — and watched a non-unitary matrix break normalisation.
- You used $U^{-1} = U^\dagger$ to undo a gate, and saw a single-qubit gate as a rotation of the Bloch sphere.

You can now *operate* on quantum states, not only describe them — a genuine step up. Everything computational the course does is built from these rotations.

## What's next

Time to name names. In `01/07_pauli_gates` we meet the three most important single-qubit gates — $X$, $Y$, $Z$ — and uncover their double life: they are gates you apply *and* the observables whose averages are the Bloch coordinates.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 1.3, 4.2, Cambridge University Press (2010).
- J. J. Sakurai & J. Napolitano, *Modern Quantum Mechanics*, ch. 1.5 (unitary operators), 3rd ed., Cambridge University Press (2020).

**Previous:** `notebooks/01_QuantumFoundations/05_inner_product_and_dirac.ipynb` · **Next:** `notebooks/01_QuantumFoundations/07_pauli_gates.ipynb`